# 人工ナノシート領域分割デモ
# Synthetic Nanosheet Instance Segmentation Demo

---

## このノートブックで学ぶこと

このノートブックでは、人工的に生成したナノシート風画像を使って、画像領域分割の基本的な流れを体験します。

扱うタスクは、ナノシート画像中の個々のシートを分けて検出する **instance segmentation** です。

このデモでは、次の2種類の方法を比較します。

### ゼロショット分割ベースライン
学習を行わず、画像処理（適応的閾値処理・モルフォロジー・連結成分分析）によって領域を抽出する方法です。
SAM（Segment Anything Model）のようなViTベースの基盤モデルと同じ「学習なし」のアイデアを、軽量に再現しています。

### YOLO-seg（合成データで学習済み）
人工ナノシート画像100枚とその正解マスクを使って、YOLO-seg（YOLOv11s-seg）をインスタンス分割用に学習したモデルです。
このデモでは、計算時間を短くするため、あらかじめ学習・推論した予測結果（`demo_assets/`）を使います。

---

**重要：** このpublicデモには、実際の実験TEM画像、手動アノテーション、未公開の実験データは含まれていません。
すべての画像、マスク、モデルweights、予測結果は、教材用に生成した人工データに基づいています。

---

### 学ぶ流れ

1. 人工データを作る（生成の仕組みを理解する）
2. 正解マスクを確認する
3. ゼロショット分割の結果を見る
4. YOLO-seg（学習済みモデル）の結果を見る
5. 評価指標を計算する
6. どの方法がよいかを比較する

実際の研究では、ここからさらに実験データへの転移（sim2real gap）、アノテーションの不完全性、重なり領域の扱い、評価指標の選び方などが重要になります。

## Step 1: GitHubから教材を取得する

In [ ]:
import os

repo_dir = "/content/nanosheet-segmentation-colab-demo"

if not os.path.exists(repo_dir):
    !git clone https://github.com/fanfanfuzzy/nanosheet-segmentation-colab-demo.git
else:
    print("Repository already exists. Skipping clone.")

%cd /content/nanosheet-segmentation-colab-demo

## Step 2: 必要なライブラリをインストールする

In [ ]:
!pip install -r requirements.txt -q

## Step 3: 人工ナノシート画像を生成する

ここでは、物理モデル（Beer-Lambert則に基づく透過率モデル）を使って、
ナノシート風の合成画像を生成します。

難易度設定（`configs/`）によって、ノイズ・コントラスト・重なり密度が変わります。
- `synthetic_easy.yaml`: 高コントラスト、低ノイズ
- `synthetic_mid.yaml`: 中程度
- `synthetic_hard.yaml`: 低コントラスト、高ノイズ、密な重なり

In [ ]:
!python src/generate_synthetic_nanosheets.py \
    --config configs/synthetic_mid.yaml \
    --num-images 10 \
    --output-dir outputs/synthetic_demo

## Step 4: 正解マスクを可視化する

生成した画像と、各ナノシートの正解マスク（instance mask）を重ねて表示します。
色が異なる領域が、それぞれ別のナノシートです。

In [ ]:
!python src/visualize_dataset.py \
    --input-dir outputs/synthetic_demo \
    --output-dir outputs/figures

In [ ]:
from IPython.display import Image, display
display(Image("outputs/figures/sample_dataset.png"))

## Step 5: SAM（Segment Anything Model）の結果を見る

ここでは、Meta AI が開発した **SAM（Segment Anything Model）** を使ってナノシート領域を抽出します。

SAM は ViT-H（Vision Transformer - Huge）をバックボーンとした基盤モデルで、
任意の画像に対してゼロショットでセグメンテーションマスクを生成できます。

ここでは **Automatic Mask Generator（AMG）** モードを使用しています：
- 画像全体に均等にグリッドポイント（32x32）を配置
- 各ポイントからマスク候補を生成
- IoU予測スコアと安定性スコアでフィルタリング

SAM は汎用的なセグメンテーションモデルであり、ナノシートに特化した学習は行っていません。
そのため、ナノシートを正確にインスタンス分割できるかは未知数です。

**注意：** SAM推論には GPU が必要なため、事前に計算した結果を使用します。

In [ ]:
# Load pre-computed SAM (ViT-H) AMG predictions
import shutil, os
os.makedirs("outputs/predictions_sam", exist_ok=True)
for f in sorted(os.listdir("demo_assets/predictions_sam")):
    if f.endswith(".npy"):
        shutil.copy2(
            f"demo_assets/predictions_sam/{f}",
            f"outputs/predictions_sam/{f}"
        )
print(f"Loaded {len(os.listdir('outputs/predictions_sam'))} SAM predictions")

## Step 6: YOLO-seg（学習済みモデル）の結果を見る

ここでは、合成データ100枚で学習した **YOLO-seg（YOLOv11s-seg）** の予測結果を読み込みます。

学習の流れ：
1. 合成ナノシート画像100枚（train）を生成
2. YOLO-seg形式（画像 + ポリゴンラベル）に変換
3. YOLOv11s-seg を150エポック学習（GPU使用、early stopping適用）
4. 学習済みモデルで合成テスト画像10枚を推論

このデモでは計算時間を短縮するため、あらかじめ推論した予測結果を使用します。
モデルweightsは `models/yolo11s-seg-nanosheet.pt` に含まれています。

**注意：** このモデルは合成データのみで学習しています。実験データは一切使用していません。

In [ ]:
# Copy pre-computed YOLO-seg predictions
import shutil, os
os.makedirs("outputs/predictions_yolo_trained", exist_ok=True)
for f in sorted(os.listdir("demo_assets/predictions_yolo_trained")):
    if f.endswith(".npy"):
        shutil.copy2(
            f"demo_assets/predictions_yolo_trained/{f}",
            f"outputs/predictions_yolo_trained/{f}"
        )
print(f"Loaded {len(os.listdir('outputs/predictions_yolo_trained'))} YOLO-seg predictions")

## Step 7: 評価指標を計算する

両方の手法の予測結果を、正解マスクと比較して評価します。

### 評価指標の意味

| 指標 | 意味 |
|------|------|
| `instance_recall` | 正解のナノシートのうち、モデルが見つけられた割合 |
| `instance_precision` | モデルが出した予測のうち、正解だった割合 |
| `instance_f1` | RecallとPrecisionの調和平均 |
| `mean_best_iou` | 各正解に対して最もよく重なる予測とのIoUの平均 |
| `semantic_iou` | 個体を区別せず、前景全体としてのIoU |

**注意：** 実データで正解アノテーションが一部しかない場合、precisionは低く見積もられることがあります。
モデルが本当に存在するナノシートを見つけても、GTに描かれていなければfalse positive扱いになるからです。

このデモでは合成データの完全なGTを使っているため、この問題は発生しません。
しかし実際の研究では、instance_recall と mean_best_iou を主指標として読むことが重要です。

In [ ]:
# Evaluate SAM (ViT-H AMG)
!python src/evaluate_predictions.py \
    --gt-dir demo_assets/ground_truth \
    --pred-dir outputs/predictions_sam \
    --method-name sam_vit_h \
    --output outputs/metrics_sam.csv

In [ ]:
# Evaluate YOLO-seg trained model
!python src/evaluate_predictions.py \
    --gt-dir demo_assets/ground_truth \
    --pred-dir outputs/predictions_yolo_trained \
    --method-name yolo_trained \
    --output outputs/metrics_yolo.csv

## Step 8: ゼロショットとYOLO-segを比較する

ここが本デモの核心です。

SAM（汎用基盤モデル）は学習データなしでも使える便利な方法です。
しかし、タスクに合わせた合成ラベルが用意できれば、YOLO-segのような学習済みモデルの方が
インスタンスレベルの分割性能を向上させることができます。

以下のスクリプトで、SAMとYOLO-segを同じデータ・同じ指標で直接比較します。

**注意：** このpublicデモには、実際の実験データは含まれていません。
すべて合成データに基づく比較です。

In [ ]:
!python src/compare_zero_shot_vs_trained.py \
    --gt-dir demo_assets/ground_truth \
    --zero-shot-dir outputs/predictions_sam \
    --trained-dir outputs/predictions_yolo_trained \
    --output-csv outputs/comparison_metrics.csv \
    --output-fig outputs/comparison_barplot.png

In [ ]:
import pandas as pd
from IPython.display import Image, display

# Show comparison table
df = pd.read_csv("outputs/comparison_metrics.csv")
display(df)

# Show comparison bar chart
display(Image("outputs/comparison_barplot.png"))

## Step 9: テスト画像ごとの可視化比較

各テスト画像について、元画像・正解マスク・ゼロショット予測・YOLO-seg予測を並べて表示します。

インスタンスごとに色分けされたオーバーレイで、各手法の検出状況を直感的に比較できます。

In [ ]:
!python src/visualize_comparison.py \
    --image-dir demo_assets/test_images \
    --gt-dir demo_assets/ground_truth \
    --zero-shot-dir demo_assets/predictions_sam \
    --pretrained-dir demo_assets/predictions_yolo_pretrained \
    --trained-dir outputs/predictions_yolo_trained \
    --output-dir outputs/comparison_visual

In [ ]:
from IPython.display import Image, display
from pathlib import Path

# Show per-image comparison grids
for img_path in sorted(Path("outputs/comparison_visual").glob("comparison_[0-9]*.png")):
    print(f"\n--- {img_path.stem} ---")
    display(Image(str(img_path), width=900))

## Step 10: 発展 — YOLOの短時間学習を試す（任意）

---

**このセルは実行しなくてもワークショップは完了します。**

GPUランタイムが利用可能な場合のみ、短いYOLO学習を体験できます。
実行には数分かかる場合があります。

```
ランタイム → ランタイムのタイプを変更 → GPU を選択
```

In [ ]:
# Optional: run a very short YOLO-seg training demo.
# This cell requires a GPU runtime and may take a few minutes.
# Uncomment the lines below to try.

# !pip install ultralytics -q
# !yolo segment train \
#     model=yolo11n-seg.pt \
#     data=demo_assets/yolo_dataset.yaml \
#     epochs=3 \
#     imgsz=512 \
#     batch=4

## Step 11: まとめ

このデモでは、以下を学びました：

1. **人工ナノシート画像の生成** — 物理モデルに基づく合成データ作成
2. **ゼロショット分割** — 学習なしで候補マスクを生成する方法（SAMのアイデアを簡易再現）
3. **YOLO-seg（学習済みモデル）** — 合成データ100枚で学習したYOLOv11s-segによるインスタンス分割
4. **評価指標** — instance recall, precision, IoU による定量評価
5. **ゼロショット vs YOLO-segの直接比較** — 棒グラフによる性能比較
6. **テスト画像ごとの可視化** — 元画像・GT・ゼロショット・YOLO-segの並列表示で直感的に理解

---

### 実際の研究では

実際の研究では、合成データで高性能だったモデルでも、実験画像に適用すると性能が低下する場合があります。
この差は **sim2real gap** と呼ばれます。

本publicデモでは、実験データそのものは公開せず、合成データを使って同じ評価の流れだけを体験しました。

---

### AI支援コーディングについて

このリポジトリの一部は、AIコーディングアシスタント（Devin）を使って作成されています。

重要なのは、AIにコードを書かせることよりも、
**AIが作った変更をGitHub上で確認し、安全に取り込むこと**です。

Pull Requestの「Files changed」タブで差分を確認する習慣をつけましょう。